# BestShot — Single Production Notebook (Kubernetes-First)

Run this notebook from **Experiment > Jupyter Interface** on chameleoncloud.org.

Before running:
1. You must have an **active** `gpu_mi100` lease on CHI@TACC
2. Your SSH key must be uploaded to CHI@TACC

This is the **only production operator notebook**.
Production flow is Kubernetes-first and automated through `infra/bootstrap.sh`.

Run all cells top to bottom. At the end you will have a floating IP and a fully bootstrapped K8s cluster.

## 1. Connect to your project

In [ ]:
from chi import server, context, lease, network
import os

context.version = "1.0"
context.choose_project()               # pick your CHI-XXXXXX project from the dropdown
context.choose_site(default="CHI@TACC")

## 2. Get your active lease

Change `YOUR_LEASE_NAME` to whatever you named your lease on Chameleon (e.g. `bestshot-serving`).

In [ ]:
LEASE_NAME = "serve_proj19_testing_lease"   # change this

l = lease.get_lease(LEASE_NAME)
l.show()   # status should say ACTIVE

## 3. Launch the bare metal server

Uses the ROCm-compatible Ubuntu image (required for AMD MI100 GPU).

**Note:** if you already have a server with this name in ERROR state, delete it first in the Horizon GUI before running this cell.

In [ ]:
username = os.getenv("USER")

s = server.Server(
    f"node-bestshot-{username}",
    reservation_id=l.node_reservations[0]["id"],
    image_name="CC-Ubuntu24.04-ROCm"
)
s.submit(idempotent=True)
print("Server submitted — waiting for it to become active...")

## 4. Configure security groups

Opens inbound TCP for SSH, Immich and MLflow NodePorts, and **Grafana (30300)** / **Prometheus (30900)** so you can open the monitoring UIs in a browser at `http://<floating-ip>:<port>` (same ports as `infra/k8s/monitoring/monitoring-stack.yaml`).

If your project allows it, tighten rules in Horizon (e.g. source = your IP /32) instead of world-open ingress.

In [ ]:
security_groups = [
    {"name": "allow-ssh", "port": 22, "description": "Enable SSH traffic on TCP port 22"},
    {"name": "allow-30283", "port": 30283, "description": "Enable Immich NodePort access on TCP port 30283"},
    {"name": "allow-30500", "port": 30500, "description": "Enable MLflow NodePort access on TCP port 30500"},
    {"name": "allow-30300", "port": 30300, "description": "Enable Grafana NodePort (bestshot-platform/grafana)"},
    {"name": "allow-30900", "port": 30900, "description": "Enable Prometheus NodePort (bestshot-platform/prometheus)"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        "name": sg["name"],
        "description": sg["description"],
    })
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"Updated security groups: {[sg['name'] for sg in security_groups]}")

## 5. Assign a floating IP so you can SSH in

In [ ]:
s.associate_floating_ip()

In [ ]:
s.refresh()
s.check_connectivity()

In [ ]:
s.refresh()
s.show(type="widget")   # note the floating IP in the Addresses row

## 6. Add team SSH keys

In [ ]:
s.execute("""cat >> ~/.ssh/authorized_keys << 'EOF'
# Anokhi
ssh-rsa AAAAB3NzaC1yc2EAAAADAQABAAABgQD2GL6pig+qGS9zhy/HvWs0CewQ5qqsg1QiG80Wxpko4URU4WeavVkmOl3iTPXYapXRbKvGXcDhGtgjO0OwYEfJK8Lhie86U4vavkcYn4bX1GsXcC6HWfUCu0rUVnMCDvq1pdifWb0fXO00OtaoImzg7vEi3BKltwzYJjaxSoG5xx+MEYniMh4lgIUF8nJ60xEX+GXn37TzglWkOFwRg+h7mhJN31BpZVpM1c3UT2MC1grwlbFo7YQl8NeKZiN9+HSBKokIBDO1QNximDnZ+Vl0ZgKojUSxmOAEvJYSio5I/wmz9lOULkOcKmHpOGGrGQtDg8BeWQF44WOCyJZAMYDz/X5yCo1ptt+lDJLc/xBp/agX6A7yTQIMBwBYcbX352H1p8cfWJ9ZltZ0PcWvo9thrLJykCL/cyIEE1drYpkCWl2cgM1VOHf1itqfb8CBU4Y099Xic5nujMsFhfYTIUf7VFCcvPViO0ce4EihKA2G2ptL5/waYIV7fGLF+OZxAmc= anokhimehta@Anokhis-Air.lan

# Anurag  
# ssh-rsa AAAA...

# Dhanush
ssh-rsa AAAAB3NzaC1yc2EAAAADAQABAAABgQDnttdrVF51qay5h3qVBIQ6oMDr+jNlrslZD76agx/H2R5zMxNqOnLDJyL4SR242M+YhLRDrn8KF2tBzHBPFHAA8OWgO8FoXopa/qY7mQjealusbSr4uGWxXJV5XZ5L6iY3DAURpUdgC+AY0RHqejd4KckXBKUs6ZOAScT3sFFC3C+m0WUm3jYB2iw2iSivevde1RxoXdQ2F9FBvz1c5DNL9Q2WPg0C2QjUTYjDw6nnCIjmx7vUedUFN/Wjxpn/l1DN4pVOhkoj7uHVwG6hMs0WMx9J0CZ5FvX+VBAKldKx87WCVKQ8rrxTw33/vEUIN/6rxOW/dwDctmEgvu7oP3cMzf+mvPQnNCbUYAjmvFbkJ8iKgZXj66uJvuzbzzHfVce7/BJMto5EKGSHBxoqZshQL4IN4nAP7LLuV7X6tVbPo9XO+WUvZGa2fIMTM8nDFG54n/TbMiew1THA8gAob6lVgRsrlRYmoQJK1HpRitsvq6vokTc3MopAsLY5g7W7UGM= dhanu@acer

EOF
""")
print("Team SSH keys added!")

## 7. Sync repository on node (idempotent)

In [ ]:
s.execute("""
if [ -d ~/bestshot/.git ]; then
  cd ~/bestshot && git pull
else
  git clone https://github.com/anokhimehta/bestshot.git ~/bestshot
fi
""")
print("Repository is ready at ~/bestshot")

## 8. Configure required environment variables (Kubernetes bootstrap)

Note: `IMMICH_API_KEY` is optional during first bootstrap and can be added later after Immich UI is available.

In [ ]:
ENV = {
    "OS_AUTH_URL": "https://chi.tacc.chameleoncloud.org:5000/v3",
    "OS_APPLICATION_CREDENTIAL_ID": "<id>",
    "OS_APPLICATION_CREDENTIAL_SECRET": "<secret>",
    "OS_REGION_NAME": "CHI@TACC",
    "BUCKET_NAME": "<swift_bucket>",
    "AWS_ACCESS_KEY_ID": "<s3_access_key>",
    "AWS_SECRET_ACCESS_KEY": "<s3_secret_key>",
    "IMMICH_DB_PASSWORD": "<db_password>",
}

cmd = "\n".join([f'export {k}="{v}"' for k, v in ENV.items()])
cmd += "\nexport MLFLOW_TRACKING_URI=\"http://$(curl -s ifconfig.me):30500\"\n"
cmd += "\n" + "cd ~/bestshot && bash infra/bootstrap.sh"

s.execute(cmd)
print("Bootstrap started with required variables.")
print("If IMMICH_API_KEY is not set yet, add it later and restart sidecar:")
print("kubectl create secret generic immich-sidecar-secret --from-literal=IMMICH_API_KEY='<immich_api_key>' -n bestshot-app --dry-run=client -o yaml | kubectl apply -f -")
print("kubectl rollout restart deployment/immich-sidecar -n bestshot-app")

## 9. Verify Kubernetes cluster and core workloads

In [ ]:
s.execute("""
export KUBECONFIG=$HOME/.kube/config
kubectl get nodes
kubectl get pods --all-namespaces
kubectl get cronjobs -n bestshot-platform
kubectl get svc -n bestshot-platform
kubectl get svc -n bestshot-app
""")

## 10. Check external endpoints

Production endpoints after bootstrap:
- Immich UI: `http://<floating-ip>:30283`
- MLflow UI/API: `http://<floating-ip>:30500`

Note: `30500` is the canonical project entrypoint. Backing implementation can be either in-cluster MLflow or routed backend, but clients should always use this NodePort URL.

In [ ]:
s.execute("""
IP=$(curl -s ifconfig.me)
echo "Immich: http://$IP:30283"
echo "MLflow: http://$IP:30500"
""")

## 11. Optional checks (Kubernetes only)

In [ ]:
s.execute("""
export KUBECONFIG=$HOME/.kube/config
kubectl get pods -n bestshot-platform
kubectl get pods -n bestshot-app
kubectl get svc -n bestshot-platform
kubectl get svc -n bestshot-app
""")

## 12. Re-apply manifests safely (idempotent)

In [ ]:
s.execute("""
export KUBECONFIG=$HOME/.kube/config
cd ~/bestshot && git pull
kubectl apply -f infra/k8s/platform/
kubectl apply -f infra/k8s/app/
kubectl apply -f infra/k8s/environments/staging/
kubectl apply -f infra/k8s/environments/canary/
kubectl apply -f infra/k8s/environments/production/
kubectl apply -f infra/k8s/monitoring/
kubectl apply -f infra/k8s/jobs/
""")

## 13. Optional diagnostics (Kubernetes only)

In [ ]:
s.execute("""
export KUBECONFIG=$HOME/.kube/config
kubectl get all -n bestshot-platform
kubectl get all -n bestshot-app
kubectl get cronjobs -n bestshot-platform
""")

## 15. Delete Resources

In [ ]:
print("No host Docker cleanup required for production K8s path.")
print("Use kubectl delete for K8s resources when needed.")

In [ ]:
s.execute("""
export KUBECONFIG=$HOME/.kube/config
kubectl get all -n bestshot-platform
kubectl get all -n bestshot-app
""")

In [ ]:
# Shut down the bare metal server to release it back
s.delete()